# Agent Graph Explorer

이 노트북은 `2.agentic-ai-project`의 LangGraph 라우팅 에이전트를 단계적으로 탐색하기 위한 학습용 도구입니다.

**구성**
1. 경로 설정 & 환경 점검
2. Agent 그래프 빌드
3. 그래프 시각화 (인라인 PNG)
4. Session 생성
5. 데모 질의 (SQL / Web Search / RAG)
6. 내부 상태 관찰 (route, route_reason, tool result)
7. 자유 질의
8. 대화 이력 확인 / 초기화

**사전 준비**
- `python setup.py` 로 SQLite DB(`data/ecommerce.db`)와 FAISS 인덱스(`data/faiss_index/`)를 빌드해 두어야 합니다.
- 프로젝트 루트의 `.env` 파일에 `ANTHROPIC_API_KEY`, `SERPER_API_KEY`가 설정되어 있어야 합니다.

### Step 1 — 경로 설정 & 환경 점검

노트북은 `notebooks/` 하위에서 실행되므로, 프로젝트 루트를 `sys.path`에 추가하고 작업 디렉토리를 루트로 옮겨야 `agent.*`, `session`, `config` 등을 import 할 수 있고 `config.py`의 상대 경로(`data/`, `pdf_docs/`)도 일치합니다.

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD        :", os.getcwd())

PROJECT_ROOT: /Users/kmyu/Desktop/project/langchain-study/medium/2.agentic-ai-project
CWD        : /Users/kmyu/Desktop/project/langchain-study/medium/2.agentic-ai-project


In [2]:
# API 키 / 데이터 아티팩트 존재 확인 (누락 시 경고만 출력 — 노트북 커널은 중단하지 않음)
from config import (
    ANTHROPIC_API_KEY,
    OPENAI_API_KEY,
    SERPER_API_KEY,
    DB_PATH,
    FAISS_INDEX_DIR,
)

issues = []
if not (ANTHROPIC_API_KEY or OPENAI_API_KEY):
    issues.append("환경 변수 ANTHROPIC_API_KEY (또는 OPENAI_API_KEY) 가 비어 있습니다.")
if not SERPER_API_KEY:
    issues.append("환경 변수 SERPER_API_KEY 가 비어 있습니다.")
if not os.path.exists(DB_PATH):
    issues.append(f"SQLite DB 가 없습니다: {DB_PATH}  →  `python setup.py` 실행 필요")
if not os.path.exists(os.path.join(FAISS_INDEX_DIR, "index.faiss")):
    issues.append(f"FAISS 인덱스가 없습니다: {FAISS_INDEX_DIR}/index.faiss  →  `python setup.py` 실행 필요")

if issues:
    print("[WARN] 다음 항목을 확인해 주세요:")
    for line in issues:
        print("  -", line)
else:
    print("[OK] API 키와 데이터 아티팩트가 모두 준비되었습니다.")

[OK] API 키와 데이터 아티팩트가 모두 준비되었습니다.


### ⚠️ 사전 준비 — MLflow 트래킹 서버

아래 셀(`init_observability()`)은 내부적으로 `mlflow.set_tracking_uri(...)` 와 `mlflow.set_experiment(...)` 를 호출합니다. `set_experiment`은 트래킹 서버에 **실제 HTTP 요청을 보내** 실험을 조회/생성하므로, 서버가 떠 있지 않으면 `ConnectionRefusedError` 가 발생하고 urllib3 재시도 루프에서 셀이 멈춥니다.

`.env` 의 기본값은 `MLFLOW_TRACKING_URI=http://localhost:5001` 입니다 (macOS의 AirPlay Receiver가 5000번 포트를 점유하기 때문에 5001 사용).

**선택지 3가지**

1. **MLflow 서버 실행 (권장)** — 별도 터미널에서:
   ```bash
   uv run mlflow server --host 0.0.0.0 --port 5001
   # uv 미사용이면: mlflow server --host 0.0.0.0 --port 5001
   ```
   서버가 뜬 뒤 아래 셀을 실행하고, UI는 http://localhost:5001 에서 확인합니다. `@trace` 결과를 시각화하는 것이 이 모듈의 핵심이므로 학습 목적이라면 이 방법을 권장합니다.

2. **Observability 비활성화** — 트레이싱이 필요 없다면 `.env` 에 다음을 추가:
   ```
   MLFLOW_ENABLED=false
   ```
   `observability.py` 의 `ENABLED` 플래그가 `False` 가 되어 `init()` 과 `@trace` 데코레이터가 모두 패스스루로 동작합니다.

3. **서버 없이 로컬 파일/SQLite 백엔드** — 서버를 띄우지 않고 로컬에 기록만 남기고 싶다면 `.env` 의 URI를 변경:
   ```
   MLFLOW_TRACKING_URI=file:./mlruns
   # 또는
   MLFLOW_TRACKING_URI=sqlite:///mlflow.db
   ```
   나중에 UI가 필요해지면 `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001` 로 열 수 있습니다.

In [3]:
# 관측성(Observability) 초기화 — 선택 사항. MLflow tracing 등이 활성화됩니다.
from observability import init as init_observability

init_observability()

KeyboardInterrupt: 

## Step 2 — Agent 그래프 빌드

`agent/graph.py`의 `build_graph()`를 호출해 `StateGraph` 를 컴파일합니다.

그래프 토폴로지
```
START → router_node → { sql_node | rag_node | web_search_node } → synthesise_node → update_history_node → END
```

In [ ]:
from agent.graph import build_graph

compiled_graph = build_graph()
compiled_graph

## Step 3 — 그래프 시각화

LangGraph의 내장 Mermaid 렌더러로 PNG를 만들어 노트북에 인라인 표시합니다. (파일로 저장하지는 않습니다.)

In [ ]:
from IPython.display import Image, display

png_bytes = compiled_graph.get_graph().draw_mermaid_png()
display(Image(png_bytes))

## Step 4 — Session 생성

`EcommerceSession`은 컴파일된 그래프 + 대화 이력 + 턴 카운터를 캡슐화합니다. 이후 단계에서는 `session.ask(...)` 만 호출하면 됩니다.

In [ ]:
from session import EcommerceSession

session = EcommerceSession()
print("Session ID :", session.session_id)
print("Turn       :", session.turn_number)

## Step 5 — 데모 질의

라우터가 세 종류의 도구(SQL / Web Search / RAG)로 각각 분기하는지 한 셀씩 차례로 확인합니다.

### 5-1. SQL 라우트 — 주문/판매 데이터 질의

In [ ]:
answer = session.ask("How many orders were delivered successfully?")
print(answer)

### 5-2. Web Search 라우트 — 외부 최신 정보

In [ ]:
answer = session.ask("What are the latest e-commerce trends in Brazil for 2024?")
print(answer)

### 5-3. RAG 라우트 — 사내 PDF 문서 기반

In [ ]:
answer = session.ask("What does our return policy say about electronics?")
print(answer)

## Step 6 — 내부 상태 관찰

`session.ask()`는 최종 답변만 반환하지만, 학습 목적으로 한 번은 `compiled_graph.invoke()`를 직접 호출해 `route` (선택된 도구), `route_reason` (라우터의 판단 근거), 그리고 해당 도구의 원시 결과까지 살펴봅니다.

이 셀은 별도 invoke 이므로 `session.conversation_history` 에는 누적되지 않습니다.

In [ ]:
from agent.state import AgentState

initial_state: AgentState = {
    "user_message":         "Show me the top 5 product categories by total revenue.",
    "conversation_history": session.get_history(),
    "route":                "",
    "route_reason":         "",
    "sql_result":           None,
    "rag_result":           None,
    "web_search_result":    None,
    "final_answer":         "",
    "turn_number":          session.turn_number + 1,
}

final_state = compiled_graph.invoke(initial_state)

print("Route        :", final_state["route"])
print("Route reason :", final_state["route_reason"])

print("\n--- Tool result ---")
for key in ("sql_result", "rag_result", "web_search_result"):
    if final_state.get(key):
        print(f"[{key}]")
        print(final_state[key])

print("\n--- Final answer ---")
print(final_state["final_answer"])

## Step 7 — 자유 질의

아래 `question` 문자열만 바꿔서 같은 셀을 반복 실행하면 됩니다. 같은 `session` 객체를 사용하므로 이전 턴의 맥락이 유지됩니다.

In [ ]:
question = "여기에 질문을 입력하세요"

answer = session.ask(question)
print(answer)

## Step 8 — 대화 이력 확인 / 초기화

In [ ]:
session.print_history()

In [ ]:
# 필요 시 실행 — 대화 이력을 모두 비웁니다.
session.reset()